- all-foundations-of-empirical-research-spring-2026
- Assignment # 5 
- Alia Kasem

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [0]:
diamonds = spark.table("diamonds")
diamonds.display()

In [0]:
diamonds_pd = diamonds.toPandas()

In [0]:


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Diamonds Data Overview', fontsize=16)

# Top left: Count of diamonds by cut
sns.countplot(data=diamonds_pd, x='cut', order=['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
              palette='viridis', ax=axes[0, 0])
axes[0, 0].set_title('Count of Diamonds by Cut')
axes[0, 0].set_xlabel('Cut')
axes[0, 0].set_ylabel('Count')

# Top right: Box plot of price by cut
sns.boxplot(data=diamonds_pd, x='cut', y='price',
            order=['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
            palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('Price Distribution by Cut')
axes[0, 1].set_xlabel('Cut')
axes[0, 1].set_ylabel('Price ($)')

# Bottom left: Histogram of diamond prices
sns.histplot(diamonds_pd['price'], bins=50, kde=True, color='coral', ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Diamond Prices')
axes[1, 0].set_xlabel('Price ($)')
axes[1, 0].set_ylabel('Frequency')

# Bottom right: Scatter plot of carat vs price, colored by cut
sns.scatterplot(data=diamonds_pd.sample(2000), x='carat', y='price', hue='cut',
                palette='Set1', alpha=0.7, ax=axes[1, 1])
axes[1, 1].set_title('Carat vs Price by Cut (Sampled 2000 Points)')
axes[1, 1].set_xlabel('Carat')
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].legend(title='Cut')

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave space for suptitle
plt.show()

`Overall takeaway:`
The data shows that diamonds with better cuts and larger carat sizes usually cost more. Still, prices vary a lot, which suggests other factors also play a role. Most diamonds are moderately priced, but a few very expensive ones affect the overall price range.

Flight data 

In [0]:
# Flights per carrier
carrier_counts = spark.sql("""
SELECT carrier, COUNT(*) AS flights
FROM flights
GROUP BY carrier
ORDER BY flights DESC
""").toPandas()

# Average delay by month
monthly_delay = spark.sql("""
SELECT month, AVG(arr_delay) AS avg_delay
FROM flights
WHERE arr_delay IS NOT NULL
GROUP BY month
ORDER BY month
""").toPandas()

# Departure delay distribution
delay_hist = spark.sql("""
SELECT dep_delay
FROM flights
WHERE dep_delay BETWEEN -60 AND 120
""").toPandas()

# Top 5 carriers arrival delays
top5 = spark.sql("""
SELECT carrier, arr_delay
FROM flights
WHERE arr_delay BETWEEN -60 AND 120
AND carrier IN (
    SELECT carrier
    FROM flights
    GROUP BY carrier
    ORDER BY COUNT(*) DESC
    LIMIT 5
)
""").toPandas()

In [0]:
sns.set_style("whitegrid")

fig, axes = plt.subplots(2,2, figsize=(15,10))

# Flights per carrier
sns.barplot(
    x=carrier_counts['carrier'],
    y=carrier_counts['flights'],
    palette="viridis",
    ax=axes[0,0]
)
axes[0,0].set_title("Flights per Carrier", fontsize=14)
axes[0,0].set_xlabel("Carrier")
axes[0,0].set_ylabel("Number of Flights")

# Average delay by month
sns.lineplot(
    x=monthly_delay['month'],
    y=monthly_delay['avg_delay'],
    marker="o",
    color="crimson",
    ax=axes[0,1]
)
axes[0,1].set_title("Average Arrival Delay by Month", fontsize=14)
axes[0,1].set_xlabel("Month")
axes[0,1].set_ylabel("Average Delay (minutes)")

# Departure delay distribution
sns.histplot(
    delay_hist['dep_delay'],
    bins=40,
    kde=True,
    color="teal",
    ax=axes[1,0]
)
axes[1,0].set_title("Departure Delay Distribution", fontsize=14)
axes[1,0].set_xlabel("Departure Delay")
axes[1,0].set_ylabel("Frequency")

# Arrival delay by carrier
sns.boxplot(
    x="carrier",
    y="arr_delay",
    data=top5,
    palette="Set2",
    ax=axes[1,1]
)
axes[1,1].set_title("Arrival Delay by Top 5 Carriers", fontsize=14)
axes[1,1].set_xlabel("Carrier")
axes[1,1].set_ylabel("Arrival Delay")

plt.suptitle("Flight Delay Overview", fontsize=18)

plt.tight_layout()
plt.show()

`Overall Insight:`
The analysis shows that most flights leave and arrive close to their scheduled times, but delay patterns change depending on the month and airline. Seasonal factors and how airlines operate probably affect these patterns, and all major carriers sometimes have extreme delays.